In [7]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import imageio

#############################
# Global random map
#############################
np.random.seed(0)
global_random_map = np.random.rand(512, 512)  # Pre-generate up to 512×512

#############################
# Helper: fill region with rough terrain
#############################
def fill_with_rough_terrain(heightmap, x_start, x_end, y_start, y_end, roughness):
    region_h = y_end - y_start
    region_w = x_end - x_start
    rough_chunk = global_random_map[:region_h, :region_w]
    heightmap[y_start:y_end, x_start:x_end] = rough_chunk * roughness

#############################
# Helper: add one 2D stair rectangle
#############################
def fill_one_stair_rectangle(heightmap, occupancy,
                             x_start, x_end, y_start, y_end,
                             rect_x, rect_y, rect_w, rect_h,
                             stair_height):
    rect_x2 = min(rect_x + rect_w, x_end)
    rect_y2 = min(rect_y + rect_h, y_end)
    if rect_x2 <= rect_x or rect_y2 <= rect_y:
        return False  # out of bounds

    # Local occupancy indices
    occ_x0 = rect_x - x_start
    occ_x1 = rect_x2 - x_start
    occ_y0 = rect_y - y_start
    occ_y1 = rect_y2 - y_start
    region_occupied = occupancy[occ_y0:occ_y1, occ_x0:occ_x1]
    if np.any(region_occupied):
        return False  # overlap => skip

    # Random steps
    n_steps_x = np.random.randint(2, 6)  # 2..5
    n_steps_y = np.random.randint(2, 6)  # 2..5

    total_w = rect_x2 - rect_x
    total_h = rect_y2 - rect_y
    step_w = total_w / float(n_steps_x)
    step_h = total_h / float(n_steps_y)

    for iy in range(n_steps_y):
        frac_down = iy / (n_steps_y - 1)  # 0 => top, 1 => bottom
        for ix in range(n_steps_x):
            frac_right = ix / (n_steps_x - 1)  # 0 => left, 1 => right
            cell_height = stair_height * frac_right * (1.0 - frac_down)

            x0 = int(rect_x + ix * step_w)
            x1 = int(rect_x + (ix+1) * step_w)
            y0 = int(rect_y + iy * step_h)
            y1 = int(rect_y + (iy+1) * step_h)
            heightmap[y0:y1, x0:x1] = cell_height

    # Mark occupancy
    occupancy[occ_y0:occ_y1, occ_x0:occ_x1] = True
    return True

#############################
# Main fill_staircase_region
#############################
def fill_staircase_region(heightmap,
                          x_start, x_end, y_start, y_end,
                          coverage, stair_height,
                          stair_length, stair_width,
                          leftover_roughness):
    # Fill entire region with rough terrain first
    fill_with_rough_terrain(heightmap, x_start, x_end, y_start, y_end, leftover_roughness)

    region_w = x_end - x_start
    region_h = y_end - y_start
    region_area = region_w * region_h
    target_stair_area = coverage * region_area

    occupancy = np.zeros((region_h, region_w), dtype=bool)

    placed_area = 0.0
    max_attempts = 1000
    for _ in range(max_attempts):
        if placed_area >= target_stair_area:
            break
        rx = x_start + np.random.randint(0, region_w)
        ry = y_start + np.random.randint(0, region_h)
        rect_w = int(stair_length)
        rect_h = int(stair_width)

        success = fill_one_stair_rectangle(
            heightmap, occupancy,
            x_start, x_end, y_start, y_end,
            rx, ry, rect_w, rect_h, stair_height
        )
        if success:
            placed_area += rect_w * rect_h

#############################
# Master heightmap generator
#############################
def generate_heightmap(size, n_regions, region_info):
    """
    n_regions in {1,4,9} => 1x1, 2x2, 3x3 grid.
    region_info is a list of dicts describing each region.
    """
    heightmap = np.zeros((size, size), dtype=np.float32)
    grid_size = int(np.sqrt(n_regions))
    sub_size = size // grid_size

    for idx in range(n_regions):
        info = region_info[idx]
        row = idx // grid_size
        col = idx % grid_size

        y_start = row * sub_size
        y_end   = (row+1)*sub_size if row < grid_size-1 else size
        x_start = col * sub_size
        x_end   = (col+1)*sub_size if col < grid_size-1 else size

        if info['type'] == 'Rough Terrain':
            fill_with_rough_terrain(heightmap, x_start, x_end, y_start, y_end,
                                    info['roughness'])
        else:
            fill_staircase_region(heightmap,
                                  x_start, x_end, y_start, y_end,
                                  coverage=info['coverage'],
                                  stair_height=info['stair_height'],
                                  stair_length=info['stair_length'],
                                  stair_width=info['stair_width'],
                                  leftover_roughness=info['leftover_roughness'])
    return heightmap

#############################
# Global to store last heightmap for saving
#############################
last_heightmap = None

#############################
# Widget creation
#############################

# 1) Size slider: from 256 to 512 in steps of 16
size_slider = widgets.IntSlider(
    value=256, min=256, max=512, step=16,
    description='Size', continuous_update=False
)

# 2) Regions dropdown
n_regions_dropdown = widgets.Dropdown(
    options=[1, 4, 9],
    value=1,
    description='Regions'
)

# We'll create 9 "accordions," one for each region
region_accordions = []
region_type_widgets = []
region_widgets = {
    'roughness': [],
    'coverage': [],
    'stair_height': [],
    'stair_length': [],
    'stair_width': [],
    'leftover_roughness': []
}

for i in range(9):
    # Type dropdown
    rt = widgets.Dropdown(
        options=["Rough Terrain", "Staircase"],
        value="Rough Terrain",
        description="Type",
        style={'description_width': '70px'},
    )
    region_type_widgets.append(rt)

    # Roughness
    w_r = widgets.FloatSlider(
        value=0.3, min=0.0, max=1.0, step=0.05,
        description="Rough",
        style={'description_width': '70px'}
    )
    region_widgets['roughness'].append(w_r)

    # Stair coverage
    w_c = widgets.FloatSlider(
        value=0.5, min=0.0, max=1.0, step=0.05,
        description="Coverage",
        style={'description_width': '70px'}
    )
    region_widgets['coverage'].append(w_c)

    # Stair height
    w_h = widgets.FloatSlider(
        value=0.8, min=0.0, max=2.0, step=0.1,
        description="Height",
        style={'description_width': '70px'}
    )
    region_widgets['stair_height'].append(w_h)

    # Stair length
    w_len = widgets.IntSlider(
        value=20, min=5, max=200, step=5,
        description="Len",
        style={'description_width': '70px'}
    )
    region_widgets['stair_length'].append(w_len)

    # Stair width
    w_wid = widgets.IntSlider(
        value=20, min=5, max=200, step=5,
        description="Width",
        style={'description_width': '70px'}
    )
    region_widgets['stair_width'].append(w_wid)

    # Leftover roughness
    w_lor = widgets.FloatSlider(
        value=0.3, min=0.0, max=1.0, step=0.05,
        description="LeftoverR",
        style={'description_width': '70px'}
    )
    region_widgets['leftover_roughness'].append(w_lor)

    # We'll place them all in a VBox, but only enable the relevant ones
    region_vbox = widgets.VBox([
        rt,
        w_r,            # roughness
        w_c, w_h, w_len, w_wid, w_lor
    ])

    # Put this vbox in an Accordion
    acc = widgets.Accordion([region_vbox])
    acc.set_title(0, f"Region {i+1}")
    acc.selected_index = None  # collapse by default
    region_accordions.append(acc)

#############################
# Visibility logic
#############################
def update_visibility():
    # How many regions?
    n = n_regions_dropdown.value

    for i in range(9):
        if i < n:
            region_accordions[i].layout.display = ''  # show
            # Which type?
            rtype = region_type_widgets[i].value
            if rtype == "Rough Terrain":
                # Enable roughness, disable the others
                region_widgets['roughness'][i].disabled = False
                region_widgets['coverage'][i].disabled = True
                region_widgets['stair_height'][i].disabled = True
                region_widgets['stair_length'][i].disabled = True
                region_widgets['stair_width'][i].disabled = True
                region_widgets['leftover_roughness'][i].disabled = True
            else:
                # Staircase => disable roughness, enable the rest
                region_widgets['roughness'][i].disabled = True
                region_widgets['coverage'][i].disabled = False
                region_widgets['stair_height'][i].disabled = False
                region_widgets['stair_length'][i].disabled = False
                region_widgets['stair_width'][i].disabled = False
                region_widgets['leftover_roughness'][i].disabled = False
        else:
            region_accordions[i].layout.display = 'none'  # hide entire region

def on_n_regions_change(change):
    if change['name'] == 'value':
        update_visibility()

n_regions_dropdown.observe(on_n_regions_change, names='value')

def on_region_type_change(change):
    if change['name'] == 'value':
        update_visibility()

for rt in region_type_widgets:
    rt.observe(on_region_type_change, names='value')

#############################
# Update & Plot
#############################
output = widgets.Output()

def generate_and_plot(_=None):
    global last_heightmap
    with output:
        output.clear_output()

        size = size_slider.value
        n_regions = n_regions_dropdown.value

        # Build region info
        region_info = []
        for i in range(n_regions):
            rtype = region_type_widgets[i].value
            if rtype == "Rough Terrain":
                r_rough = region_widgets['roughness'][i].value
                region_info.append({
                    'type': 'Rough Terrain',
                    'roughness': r_rough
                })
            else:
                cov  = region_widgets['coverage'][i].value
                sth  = region_widgets['stair_height'][i].value
                slen = region_widgets['stair_length'][i].value
                swid = region_widgets['stair_width'][i].value
                lor  = region_widgets['leftover_roughness'][i].value
                region_info.append({
                    'type': 'Staircase',
                    'coverage': cov,
                    'stair_height': sth,
                    'stair_length': slen,
                    'stair_width': swid,
                    'leftover_roughness': lor
                })

        hm = generate_heightmap(size, n_regions, region_info)
        last_heightmap = hm

        plt.figure(figsize=(5,5))
        plt.title(f"Heightmap ({n_regions} region{'s' if n_regions>1 else ''}, size={size})")
        plt.imshow(hm, origin='lower', cmap='gray')
        plt.colorbar(label="Height")
        plt.show()

#############################
# Buttons
#############################
update_button = widgets.Button(description="Generate Heightmap")
update_button.on_click(generate_and_plot)

save_button = widgets.Button(description="Save Heightmap")

def on_save(_):
    global last_heightmap
    if last_heightmap is None:
        print("No heightmap generated yet. Please click Generate first.")
        return

    # Save as PNG
    png_data = (last_heightmap * 255).astype(np.uint8)
    imageio.imwrite("heightmap.png", png_data)
    print("Saved 'heightmap.png'.")

save_button.on_click(on_save)

#############################
# Layout
#############################
acc_box = widgets.VBox(region_accordions)
top_row = widgets.HBox([size_slider, n_regions_dropdown])
btn_row = widgets.HBox([update_button, save_button])
ui = widgets.VBox([top_row, acc_box, btn_row, output])

# Initialize visibility
update_visibility()

display(ui)
